# 04b. TabPFN on Kaggle

Run for `04a_tabpfn.ipynb`, executed on Kaggle (Tesla T4, 16 GB) because the laptop GPU (6 GB) ran out of memory and overheated. It installs `bankguard` from GitHub and TabPFN 10.2.0, reads the Prior Labs licence token from a Kaggle Secret, builds the same 22 features, and uses TabPFN-3.5-Fast with the context cached on the GPU (`fit_mode="fit_with_cache"`). Context sizes of 10,000 to 200,000 rows are probed for speed and memory, and the largest that fits is used to score all of validation. The context size is chosen on memory alone, never on validation score.

In [1]:
import sys

import torch
import transformers

print("python:", sys.version.split()[0])
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available(), "| gpus:", torch.cuda.device_count())
print("gpu:", torch.cuda.get_device_name(0))
print("transformers:", transformers.__version__)
!free -g | head -2

python: 3.13.16
torch: 2.11.0+cu130 | cuda: True | gpus: 2
gpu: Tesla T4
transformers: 5.20.2
               total        used        free      shared  buff/cache   available
Mem:              31           1          24           0           5          29


In [2]:
!pip install --no-deps --force-reinstall git+https://github.com/sap6011/bank-guard.git

  Cloning https://github.com/sap6011/bank-guard.git to /tmp/pip-req-build-dzdds7qr
  Running command git clone --filter=blob:none --quiet https://github.com/sap6011/bank-guard.git /tmp/pip-req-build-dzdds7qr
  Resolved https://github.com/sap6011/bank-guard.git to commit ddb2c058b290eb409894a2d862789fe60e9c27db
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for bank-guard: filename=bank_guard-0.1.0-py3-none-any.whl size=9861 sha256=74025ea1226e6e3073db36789638a0772a0a48f0aa95e32e78cb6bb082969d96
  Stored in directory: /tmp/pip-ephem-wheel-cache-d64lu1lf/wheels/64/0a/d6/432fce36897390a68ec24ef5723db09af1dba911df99f3f0f1
Successfully built bank-guard


In [4]:
import os

from kaggle_secrets import UserSecretsClient

os.environ["TABPFN_TOKEN"] = UserSecretsClient().get_secret("TABPFN_TOKEN")
print("token set:", bool(os.environ.get("TABPFN_TOKEN")))

!pip install -q tabpfn==10.2.0
import sklearn
import tabpfn
import torch

print(tabpfn.__version__, torch.__version__, sklearn.__version__)

token set: True
10.2.0 2.11.0+cu130 1.6.1


In [5]:
from pathlib import Path

import numpy as np
import torch

from bankguard.data import load_accounts, load_transactions, split_by_date
from bankguard.features import FEATURES, build_features, fit_currency_median

DATA = Path("/kaggle/input/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml")

tx = load_transactions(DATA / "HI-Small_Trans.csv")
accounts = load_accounts(DATA / "HI-Small_accounts.csv")
splits = split_by_date(tx)
train, val = splits["train"], splits["validation"]

X = build_features(tx, accounts, fit_currency_median(train))
X_train, y_train = X.loc[train.index, FEATURES], train["is_laundering"].to_numpy()
X_val, y_val = X.loc[val.index, FEATURES], val["is_laundering"].to_numpy()
del tx, X
print(X_train.shape, y_train.sum(), X_val.shape, y_val.sum())

(3248921, 22) 2530 (965524, 22) 1036


In [6]:
import time

from tabpfn import TabPFNClassifier
from tabpfn.constants import ModelVersion

from bankguard.features import CATEGORICAL
from bankguard.model import fit_categories, gbm_frame

categories = fit_categories(X_train, CATEGORICAL)
cat_idx = [FEATURES.index(c) for c in CATEGORICAL]


def tabpfn_frame(X):
    """The 22 features as numbers; categories become fixed codes, unseen ones missing."""
    out = gbm_frame(X, FEATURES, categories)
    for c in CATEGORICAL:
        out[c] = out[c].cat.codes.replace(-1, np.nan)
    return out.astype("float32")


def context_sample(n_total, seed=42):
    """Every laundering row plus random normal rows, n_total rows in all."""
    rng = np.random.default_rng(seed)
    pos = np.flatnonzero(y_train == 1)
    neg = rng.choice(np.flatnonzero(y_train == 0), size=n_total - len(pos), replace=False)
    return np.sort(np.concatenate([pos, neg]))


def probe(n_context, n_predict=20_000):
    ctx = context_sample(n_context)
    clf = TabPFNClassifier.create_default_for_version(ModelVersion.V3_5_FAST)
    clf.set_params(device="cuda", categorical_features_indices=cat_idx, random_state=42,
                   fit_mode="fit_with_cache")
    idx = np.random.default_rng(1).choice(len(X_val), size=n_predict, replace=False)

    torch.cuda.reset_peak_memory_stats()
    start = time.time()
    clf.fit(tabpfn_frame(X_train.iloc[ctx]), y_train[ctx])
    fit_s = time.time() - start
    start = time.time()
    clf.predict_proba(tabpfn_frame(X_val.iloc[idx]))
    pred_s = time.time() - start
    return {
        "context": n_context,
        "fit_s": round(fit_s, 1),
        "predict_20k_s": round(pred_s, 1),
        "full_val_min": round(len(X_val) / (n_predict / pred_s) / 60),
        "peak_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2),
    }


print(probe(10_000))

tabpfn-v3.5-fast-20260909.safetensors: reconstructing file:   0%|          |  0.00B /  334MB            

tabpfn-v3.5-fast-20260909.safetensors: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/36.0 [00:00<?, ?B/s]

{'context': 10000, 'fit_s': 11.6, 'predict_20k_s': 5.4, 'full_val_min': 4, 'peak_gb': 4.54}


In [7]:
results = []
for n in [50_000, 100_000, 200_000]:
    try:
        results.append(probe(n))
        print(results[-1])
    except torch.cuda.OutOfMemoryError:
        print(f"{n:,}: out of GPU memory")
        torch.cuda.empty_cache()
        break

{'context': 50000, 'fit_s': 36.5, 'predict_20k_s': 14.5, 'full_val_min': 12, 'peak_gb': 6.88}
{'context': 100000, 'fit_s': 144.8, 'predict_20k_s': 27.2, 'full_val_min': 22, 'peak_gb': 8.81}
{'context': 200000, 'fit_s': 681.6, 'predict_20k_s': 54.0, 'full_val_min': 43, 'peak_gb': 9.21}


In [8]:
import gc
import json

from bankguard.evaluate import evaluate

gc.collect()
torch.cuda.empty_cache()

n_context = results[-1]["context"]  # the largest context that fitted in memory
ctx = context_sample(n_context)
clf = TabPFNClassifier.create_default_for_version(ModelVersion.V3_5_FAST)
clf.set_params(device="cuda", categorical_features_indices=cat_idx, random_state=42,
               fit_mode="fit_with_cache")

start = time.time()
clf.fit(tabpfn_frame(X_train.iloc[ctx]), y_train[ctx])
fit_minutes = (time.time() - start) / 60

start = time.time()
val_scores = clf.predict_proba(tabpfn_frame(X_val))[:, 1]
predict_minutes = (time.time() - start) / 60
np.save("/kaggle/working/tabpfn_val_scores.npy", val_scores)

metrics = evaluate(y_val, val_scores, f"TabPFN-3.5-Fast, context {n_context:,}")
metrics.update(context=n_context, fit_minutes=fit_minutes,
               predict_minutes=predict_minutes, probes=results)
with open("/kaggle/working/tabpfn_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print({k: v for k, v in metrics.items() if k != "probes"})

{'model': 'TabPFN-3.5-Fast, context 200,000', 'pr_auc': 0.39849839199725645, 'recall_at_0.1%_fpr': 0.3967181467181467, 'recall_at_1.0%_fpr': 0.7963320463320464, 'context': 200000, 'fit_minutes': 11.361268893877666, 'predict_minutes': 42.39625282287598}
